# 05 — ETA Model Comparison & Selection

This notebook trains and compares multiple algorithms for delivery time prediction.
The winning model is selected by lowest MAE and saved for production use.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import pandas as pd
from dabba.config import get_config
from dabba.data.loaders import load_delivery
from dabba.data.cleaning import clean_delivery
from dabba.features.delivery_features import add_delivery_features
from dabba.models.eta_model import train_and_evaluate_eta_models
from dabba.models.model_selection import comparison_to_dataframe, select_best_model

config = get_config()

## 1. Load and Prepare Data

In [ ]:
df = load_delivery(config)
df = clean_delivery(df, config)
df = add_delivery_features(df, config)

# Select features
feature_cols = [c for c in df.columns if c in [
    'haversine_distance_km', 'traffic_ordinal', 'is_festival',
    'delivery_person_age', 'delivery_person_ratings', 'vehicle_condition',
]]
feature_cols += [c for c in df.columns if c.startswith('order_hour_bucket_')]

X = df[feature_cols].fillna(0)
y = df['time_taken_min']

print(f'Features: {feature_cols}')
print(f'Training samples: {len(X)}')

## 2. Train All Models with Cross-Validation

In [ ]:
results, best = train_and_evaluate_eta_models(X, y, config)

if results:
    comparison_df = comparison_to_dataframe(results, task='eta')
    print('\n=== Model Comparison ===')
    print(comparison_df.to_string(index=False))

## 3. Visualization

In [ ]:
if results:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # MAE & RMSE bar chart
    x = range(len(comparison_df))
    width = 0.35
    axes[0].bar([i - width/2 for i in x], comparison_df['mae'], width, label='MAE', color='#2196F3')
    axes[0].bar([i + width/2 for i in x], comparison_df['rmse'], width, label='RMSE', color='#FF9800')
    axes[0].set_xlabel('Model')
    axes[0].set_ylabel('Error (minutes)')
    axes[0].set_title('ETA Model Comparison — MAE & RMSE')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(comparison_df['model'], rotation=45, ha='right')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # R² bar chart
    colors = ['#4CAF50' if r > 0.7 else '#FFC107' if r > 0.5 else '#F44336' for r in comparison_df['r2']]
    axes[1].barh(comparison_df['model'], comparison_df['r2'], color=colors)
    axes[1].set_xlabel('R² Score')
    axes[1].set_title('ETA Model Comparison — R² Score')
    axes[1].axvline(x=0.7, color='green', linestyle='--', alpha=0.5, label='Good (0.7)')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

## 4. Residual Analysis (Top 3 Models)

In [ ]:
if results:
    top3 = sorted(results, key=lambda r: r.mae)[:3]
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, result in zip(axes, top3):
        if result.predictions is not None:
            residuals = result.predictions - y.values
            ax.scatter(y.values, residuals, alpha=0.2, s=5)
            ax.axhline(y=0, color='red', linestyle='--')
            ax.set_xlabel('Actual Delivery Time (min)')
            ax.set_ylabel('Residual')
            ax.set_title(f'{result.name}\nMAE={result.mae:.2f}')
            ax.grid(alpha=0.3)
    
    plt.suptitle('Residual Plots — Top 3 Models', fontsize=14)
    plt.tight_layout()
    plt.show()

## 5. Best Model Selection

In [ ]:
best_name = select_best_model(results, metric=config.eta_metric, task='eta')
print(f'\n🏆 Best ETA Model: {best_name}')
print(f'Selection criterion: Lowest {config.eta_metric.upper()}')
print(f'\nThis model will be saved and used in the dashboard, API, and SLA logic.')

## Key Takeaways

- Document which model won and why it outperformed others
- Note any patterns in the residual plots (bias at high/low delivery times)
- The winning model is the only one wired into production